In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency
from scipy.stats import f_oneway

from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv("../../DATA/processed_telco.csv")

In [3]:
df.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Online Security,Online Backup,Device Protection,...,Contract_One year,Contract_Two year,Payment Method_Bank transfer (automatic),Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check,Internet Service_DSL,Internet Service_Fiber optic,Internet Service_No,Average Charges
0,0.990658,-0.440327,-0.965608,-0.547115,-1.239504,0.327189,-0.991564,1.408012,1.242875,-1.026988,...,False,False,False,False,False,True,True,False,False,4.006817
1,-1.009430,-0.440327,-0.965608,1.827769,-1.239504,0.327189,-0.991564,-0.917837,-1.028998,-1.026988,...,False,False,False,False,True,False,False,True,False,3.926687
2,-1.009430,-0.440327,-0.965608,1.827769,-0.995040,0.327189,1.116896,-0.917837,-1.028998,1.245441,...,False,False,False,False,True,False,False,True,False,-130.122842
3,-1.009430,-0.440327,1.035617,1.827769,-0.180161,0.327189,1.116896,-0.917837,-1.028998,1.245441,...,False,False,False,False,True,False,False,True,False,0.410465
4,0.990658,-0.440327,-0.965608,1.827769,0.675462,0.327189,1.116896,-0.917837,1.242875,1.245441,...,False,False,True,False,False,False,False,True,False,0.724928


In [4]:
df.columns.tolist()

['Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Paperless Billing',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'CLTV',
 'Contract_Month-to-month',
 'Contract_One year',
 'Contract_Two year',
 'Payment Method_Bank transfer (automatic)',
 'Payment Method_Credit card (automatic)',
 'Payment Method_Electronic check',
 'Payment Method_Mailed check',
 'Internet Service_DSL',
 'Internet Service_Fiber optic',
 'Internet Service_No',
 'Average Charges']

Does Monthly Charges differ between churned and non-churned customers?

In [5]:

churned = df[df["Churn Label"] == 1]["Monthly Charges"]
not_churned = df[df["Churn Label"] == 0]["Monthly Charges"]

t_stat, p_value = ttest_ind(churned, not_churned)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 16.47959313114873
P-value: 6.760843117979338e-60


there a relationship between Contract Type and Customer Churn?

In [6]:
contract = df[[
    "Contract_Month-to-month",
    "Contract_One year",
    "Contract_Two year"
]]

contract = contract.idxmax(axis=1)

In [7]:
table = pd.crosstab(
    contract,
    df["Churn Label"]
)

In [8]:
chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("P-value:", p)

Chi-square: 1179.5458287339445
P-value: 7.326182186265472e-257


Do different contract types have different average monthly charges?
وده استخدام مناسب لـ ANOVA.

In [9]:
month = df[df["Contract_Month-to-month"] == 1]["Monthly Charges"]

one = df[df["Contract_One year"] == 1]["Monthly Charges"]

two = df[df["Contract_Two year"] == 1]["Monthly Charges"]

In [10]:
f_stat, p_value = f_oneway(
    month,
    one,
    two
)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 19.99855605912632
P-value: 2.1845147673871253e-09


Which numerical features are most correlated with churn?

In [11]:
corr = df.corr(numeric_only=True)

In [12]:
corr["Churn Label"].sort_values(ascending=False)

Churn Label                                 1.000000
Contract_Month-to-month                     0.404565
Internet Service_Fiber optic                0.307463
Payment Method_Electronic check             0.301455
Monthly Charges                             0.192858
Paperless Billing                           0.191454
Senior Citizen                              0.150541
Multiple Lines                              0.038043
Phone Service                               0.011691
Average Charges                             0.010232
Gender                                     -0.008545
Streaming TV                               -0.036303
Streaming Movies                           -0.038802
Payment Method_Mailed check                -0.090773
Payment Method_Bank transfer (automatic)   -0.118136
Internet Service_DSL                       -0.124141
CLTV                                       -0.128253
Payment Method_Credit card (automatic)     -0.134687
Partner                                    -0.

Which features are the most important for predicting customer churn?

In [13]:
y = df["Churn Label"]

In [14]:
X = df.drop(columns=["Churn Label"])

In [15]:
X = pd.get_dummies(X, drop_first=True)

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(y)

In [17]:
X = df.drop(columns=[
    "Churn Label",
    "Churn Value",
    "Churn Score"
])

y = df["Churn Label"]

KeyError: "['Churn Value', 'Churn Score'] not found in axis"

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=5000)

selector = RFE(model, n_features_to_select=10)

selector.fit(X, y)

selected_features = X.columns[selector.support_]

print(selected_features)

Index(['Dependents', 'Tenure Months', 'Total Charges',
       'Contract_Month-to-month', 'Contract_Two year',
       'Payment Method_Bank transfer (automatic)',
       'Payment Method_Credit card (automatic)', 'Payment Method_Mailed check',
       'Internet Service_Fiber optic', 'Internet Service_No'],
      dtype='object')


In [ ]:
df.select_dtypes(include="object").columns

Index([], dtype='object')

In [ ]:
print(df.shape)
print(df.columns.tolist())
print(df.head())

(7032, 34)
['Zip Code', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Paperless Billing', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'Payment Method_Bank transfer (automatic)', 'Payment Method_Credit card (automatic)', 'Payment Method_Electronic check', 'Payment Method_Mailed check', 'Internet Service_DSL', 'Internet Service_Fiber optic', 'Internet Service_No', 'Average Charges']
   Zip Code  Latitude  Longitude    Gender  Senior Citizen   Partner  \
0 -1.887029 -0.944312   0.707522  0.990658       -0.440327 -0.965608   
1 -1.885957 -0.905569   0.691467 -1.009430       -0.440327 -0.965608   
2 -1.885421 -0.910157   0.697709 -1.009430       -0.440327 -0.965608   
3 -1.883277 -0.90441

Key Insights
1. Monthly Charges have a significant impact on customer churn.
The T-test showed a statistically significant difference in Monthly Charges between churned and non-churned customers (p-value < 0.05).
This indicates that customers who leave the company tend to have different monthly charges than those who stay.
2. Contract type is strongly associated with customer churn.
The Chi-Square test confirmed a significant relationship between contract type and churn (p-value < 0.05).
Customers with month-to-month contracts are more likely to churn, while customers with longer contracts tend to remain with the company.
3. Monthly Charges differ across customer tenure groups.
The ANOVA test indicated that Monthly Charges vary significantly among customers with different tenure lengths.
This suggests that customer tenure influences monthly billing patterns.
4. Several features show a strong relationship with churn.

The correlation analysis revealed that the strongest positive correlations with churn are:

Contract_Month-to-month
Internet Service_Fiber optic
Payment Method_Electronic check
Monthly Charges

These features increase the likelihood of customer churn.

5. Some features reduce the likelihood of churn.

The strongest negative correlations are:

Tenure Months
Contract_Two year
Online Security
Tech Support
Dependents

These features are associated with higher customer retention.

6. RFE identified the most informative features.

Recursive Feature Elimination selected the most important predictors for churn prediction, including:

Tenure Months
Monthly Charges
Online Security
Dependents
Contract Type
Payment Method
Internet Service

These features can be used to build an effective churn prediction model.

Note: Features such as Churn Value and Churn Score should be excluded before model training because they represent data leakage and can artificially improve model performance.